# Introduction
The second notebook will be used as an envrioment for feature extraction and dataset generation. The dataset we will be using to extract features from is he customer-data-rich data "Dataset Name". These features will be used to enrich our current merged dataset. The aim is to enrich the data with more behavioural features in order to give the models a chance to pull out complex behavioral segments.

## - Notebook Contents:
- 0.1 - Background

- 0.2 - Importing and Initialising Dataframes

- 1.0 - Top Level Inspection
  - 1.1 - Dataset Shapes
  - 1.2  - Health Summaries

- 2.0 - Handling Null Values

- 3.0 - Amending Dataypes
  - 3.1 - Data types Overiew
  - 3.2 - Data type Format Inspection
  - 3.3 - 3.8 - Feature Datatype Conversion
  - 3.9 - Datatype Conversion Confirmation

- 4.0 - Outlier and Negatives Inspection
  - 4.1 - Negative Quantity Inspection
  - 4.2 - Outlier Inspection

- 5.0 - Standardising Formats
  - 5.1 - White Space & Letter Casing
  - 5.2 - Column Name Formatting
  - 5.3 - Datatype Compatibility Check

- 6.0 - Merging Datasets
  - 6.1 - Merging and Inspect
  - 6.2 - Duplication Removal and Inspection

- 7.0 - Notebook Conversion

## 0.1 - Imports

In [157]:
import pandas as pd

## 0.2 - Importing and Loading Datasets

In [158]:
# Load merged_df from preprocessing:
merged_df = pd.read_csv('merged_df.csv')

# Load in the third dataset for inspection
dataset3 = pd.read_csv('../raw_data/Consumer_meta.csv', skiprows = 1)

# Create copies of the original daatsets preseving the originals
df1 = dataset3.copy()
df2 = merged_df.copy()

## 1.0 - Dataset Three Feature Extraction
As mentioned this section will focus on inspecting the customer rich dataset for features that will enrich our final combined data that will be fed into the models. To the daatset will be examined for what features are currently present.

### 1.1 - Dataset Shape
Below we will display the shape of the dataset to start building an understanding of how many observations and features there is present within the data.

In [159]:
# Print the shape of dataset 3:
print("Dataset Three Shape:", df1.shape)

Dataset Three Shape: (20000, 20)


### 1.2 - Health Summary
A health summary will be deployed as a follow-on from the shape inspection. This is done to provide deeper context and understanding of the behaviour of the data we are working with. 

In [160]:
# Initlise the function: 
def health_summary(df):
    report = pd.DataFrame({
        'Data Type  ': df.dtypes,
        'Null Values  ': df.isnull().sum(),
        'Unique Values  ': df.nunique(),
        'Duplicate Values   ': df.duplicated().sum()
    })
    return report

# Print the health summaries for both dataframes:
print("-----------     Dataset Three Health Summary     ------------")
display(health_summary(df1))

-----------     Dataset Three Health Summary     ------------


,Data Type,Null Values,Unique Values,Duplicate Values
transaction_id,int64,0,20000,0
timestamp,str,0,19992,0
store_id,int64,0,45,0
city,str,0,9,0
country,str,0,4,0
store_type,str,0,3,0
product_category,str,0,6,0
product_name,str,0,43,0
unit_price,float64,0,359,0
quantity,int64,0,9,0


The health summary displays an extensive list of features (20) with multple peices of inportant information. The most important is the amount of null values. These will need to be preprocessed before extracting the features. However, it makes more sense to identify the features that will be extracted before preprocessing, as to avoid any work being carried out on features that are going to be removed.

## 2.0 - Identifying Valuable Features
This section will revolve around the process of identifying which features are suitable to be extracted and will be benificial in extending the comprehension of our current data. Determining whihc features are valuable and which are not will come down to two aspects. The first, is - is the data contained within this feature currelty present within the preprocessed merged_df? and if not, the second aspect is critically analysising whether or not the data contained within the current feature will add any value to the merged dataset?

### 2.1 - Feature List
To start, we will print a list of all the features currently contained within the dataset. 

In [161]:
# Implement a function that prints features and values:
def table_inspection(df):
    for col in df.columns:
        print(col, " - ","Example Value:", df[col].iloc[0])
# Call the above function:
print(table_inspection(df1))

transaction_id  -  Example Value: 10001
timestamp  -  Example Value: 2023-01-01 00:39:39
store_id  -  Example Value: 35
city  -  Example Value: Melbourne
country  -  Example Value: AUS
store_type  -  Example Value: Mall Kiosk
product_category  -  Example Value: Coffee
product_name  -  Example Value: Double Espresso
unit_price  -  Example Value: 3.04
quantity  -  Example Value: 1
discount_applied  -  Example Value: True
payment_method  -  Example Value: Credit Card
customer_id  -  Example Value: qiyfrwsk
customer_age_group  -  Example Value: 35-44
customer_gender  -  Example Value: Female
loyalty_member  -  Example Value: False
weather_condition  -  Example Value: Sunny
temperature_c  -  Example Value: 22.2
holiday_name  -  Example Value: New Year's Day
total_amount  -  Example Value: 2.74
None


Using the table above to identify valuable features we can come to a conclusion. The only features that pass the criteria previously set out are:

- Loyalty_member
- payment_method
- discount_applied

All other features can either be considered repeated data for what is already present within merged_df or features that do not enrich the data further - in the context of consumer behaviour. With the features identified we can drop all other non relevant ones and begin preprocessing the remaining ones.  

From the above table we can start to 'whittle down' features that do not provide any relevance to consumer segementation: 

1. 'transaction_id' - merged_df already contains transaction id's through its 'Invoice' and 'customer_id' features. This can be dropped. 

2. 'timestamp' - merged_df already contains timestamp featues that have already been parsed and split into their own numerical features, this can also be dropped. 

3. 'store_id' - merged_df does not currently contain any data like this. This feature will be kept for now but further inspection will be completed to assess the relevance of the data.

4. 'city' - Again merged_df does not contain data like this. However only one feature describing location will be needed, so this decision will be out of this feature and 'store_id' not both. 

5. 'country' - Similar to merged_df, having data that describes international locations is irrelevant for the context of this project and only serves to expand scope unnessicerilly. This feature will be dropped. 

6. 'store_type' - 

7. 'product_category' - This feature will be very useful in helping enrich consumer profiling as specific and unique products can be singled out and provide a comprehensive purchasing record. 

8. 'product_name' - This category will go hand-in-hand with 'product_category' and will also help to enrich data and produce deeper more meaningful segementation. However inspection will need to be carried oiut in order to assess how these features can be reasonably converted into numerical datatypes. 

9. 'unit_price' - This feature introduces some complication. merged_df contains a 'Price' feature already however if we are using product category and product name from this specific dataset, it might be wise to keep price as all; three of these features will be related. This feature will be kept for now but will require further inspection. 

10. 'quantity' - Again an assessment into how these products are bought might be better from this dataset than the original already found in merged_df. 

11. 'discount_applied' - keep

12. 'payment_method' - keep 

13. 'customer_id' - remove

14. 'customer_age_group' - keep

15. 'customer_gender' - keep

16. 'loyalty_member' - keep

17. 'weather_condtion' - remove

18. 'temperature_c' - remove

19. 'holiday_name' - remove

20. 'total_amount' - keep

Below code block we remove the features. 

In [162]:
# Create a variable that stores each of the pre determined features going to be dropped:
drop_features = ['transaction_id',
                'timestamp',
                'country', 
                'customer_id',
                'weather_condition',
                'temperature_c',
                'holiday_name', 
                'store_id', 
                'city', 
                'store_type', 
                'product_category', 
                'product_name', 
                'unit_price', 
                'quantity',
                'customer_age_group', 
                'customer_gender', 
                'total_amount']

# Drop the columns
df1 = df1.drop(columns=drop_features)

# Confirm feature removal was successful:
print("Remaining Features:")
print("")
print(df1.columns)

Remaining Features:

Index(['discount_applied', 'payment_method', 'loyalty_member'], dtype='str')


## 3.0 - Feature Preprocessing
From the health summary generated before we already have some information about our remaining features. Firstly there are no null values or duplicats seen within any of the observations. The issue we do have is concerned with data types. Similarly to the preprocessing comeplted for the first two datasrts, we need to convert any non-numerical features to numerical ones. Both discount Applied and Loyalty member are boolean's which means they are already binary and map directly to a numerical system. Payment method is a string, which does not nessicerily convert to numerical values striaghtfordly. The two methods we have is label encoding or one-hot encoding. One-hot encoding for this situation is the most ideal hwoever the number of values the feature has will need to be dietermined before carrying out the encoding. This is to avoid the risk of adding numerous new fetures to the dataset. The choice will come down to an assessment of what is a reasonable amount of new features to be implemented. 

### 3.1 - discount_applied & loyalty_member Binary Conversion

In [163]:
# Convert discount_applied and loyalty_member to numerical values: 
# discount_applied [YES = 1, NO = 0]:
df1['discount_applied'] = df1['discount_applied'].astype(int)
# loyalty_member [YES = 1, NO = 0]:
df1['loyalty_member'] = df1['loyalty_member'].astype(int)
# Confirmation of conversion:
# Initilise a function that produces a list of values for a given feature:
def print_values(df, feature):
    print("Unique Count:", df[feature].nunique())
    print("Value:", df[feature].unique())

# Call the function for each feature: 
print("")
print("---- discount_applied values ----")
print_values(df1, 'discount_applied')
print("")
print("---- loyalty_member values ----")
print_values(df1, 'loyalty_member')
    



---- discount_applied values ----
Unique Count: 2
Value: [1 0]

---- loyalty_member values ----
Unique Count: 2
Value: [0 1]


### 3.2 - payment_method Inspection & Conversion

In [164]:
# Print the range of values contained within payment method: 
def print_values(df, feature):
    print("Unique Count:", df[feature].nunique())
    print("Value:", df[feature].unique())

# Call the function: 
print("")
print("The value range for payment_method: ")
print_values(df1, 'payment_method')



The value range for payment_method: 
Unique Count: 4
Value: <StringArray>
['Credit Card', 'Debit Card', 'Mobile Wallet', 'Cash']
Length: 4, dtype: str


There is only 4 values - makes it resonable to implement one hot coding below
using pandas pd.get_dummies - built in function that performs one-hot coding. 

In [165]:
# Encode payement_method using Pandas: 
df1 = pd.get_dummies(df1, columns=['payment_method'])

def feature_title_list(df):
    for feature in df.columns: 
        print(feature)

# Confirm new features: 
feature_title_list(df1)

discount_applied
loyalty_member
payment_method_Cash
payment_method_Credit Card
payment_method_Debit Card
payment_method_Mobile Wallet


The results above confirm that the one-hot encoding was implemented correctly, however there is not inconsistant naming conventions for features. 

In [166]:
# Convert feature titles to snake_case: 
df1.columns = df1.columns.str.lower().str.replace(' ', '_')

# Re-confirm feature titles: 
feature_title_list(df1)

discount_applied
loyalty_member
payment_method_cash
payment_method_credit_card
payment_method_debit_card
payment_method_mobile_wallet


## 4.0 Building customer_features
customer_features is the final step of all the previous preprocessing work carried out before this point. customer_features is a new dataset that is built by combining the features from merged_df and the third dataset we have been working on in this notebook. The features in customer_features will be built by grouping merged_df by customer_id in order to generate a dataset where each row correlates to one customer. Based on the features we have been working with, customer_features will include: 

- total_spend
- average_transaction_value
- visit_frequency
- item_count
- cancellation_rate
- unique_products
- time_of_day
- time_of_week
- discount_applied
- loyalty_member
- payment_method_columns consiting of individual features: 
    - payment_method_cash
    - payment_method_credit_card
    - payment_method_debit_card
    - payment_method_mobile_wallet

similarly to the previous preprocessing steps, we will work down the list in the order that it appears.

### 4.0 - Dataset Initilisation
In order to maintain strucutre within the notebook, the code block below will be dedicated to initlising the empty dataset 'customer_features' for all subsequent code blocks to add to the dataset as they are executed. 

### 4.0 - total_spend
total spend represents the total amount of money the customer has spent in the cafe within the last 2 years. This featues is calculated by dividing total_amount by (quantity * price) per individual customer.  

### 4.0 - average_transaction_value

### 4.0 - visit_frequency

### 4.0 - item_count

### 4.0 - cancellation_rate

### 4.0 - unique_products

### 4.0 - time_of_day

### 4.0 - time_of_week

### 4.0 - discount_applied

### 4.0 - loyalty_member

### 4.0 - payment_method_columns